# DSC2107 Group 8 — itch.io Top Sellers Analysis



In [2]:
from pathlib import Path
import sqlite3, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "cleaned"
DB = ROOT / "data" / "group8_itchio.sqlite"
plt.rcParams["figure.figsize"] = (10, 5.5)

## 1. Load real collected data when available

In [3]:
csv_files = sorted(PROC.glob("*.csv"))
if not csv_files:
    csv_files = sorted(RAW.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No real CSV snapshots found in {PROC} or {RAW}")

frames = []
for file_path in csv_files:
    try:
        frames.append(pd.read_csv(file_path))
    except Exception as error:
        print("Could not read", file_path.name, error)
if not frames:
    raise ValueError("No readable real CSV snapshots were found.")

df = pd.concat(frames, ignore_index=True)
for expected, actual in {
    "url": "game_url",
    "price_text": "price",
    "genres": "genre",
    "source_page": "listing_page",
}.items():
    if expected not in df and actual in df:
        df[expected] = df[actual]
DATA_MODE = f"COLLECTED CSV FILES — {len(frames)} real snapshot(s); verify dates and provenance"
print("DATA MODE:", DATA_MODE, "| rows:", len(df))
display(df.head())

DATA MODE: COLLECTED CSV FILES — 7 real snapshot(s); verify dates and provenance | rows: 3500


,rank,title,developer,price,genre,platforms,game_url,listing_page,scraped_at,price_value,url,price_text,genres,source_page
0,1,The Choicer Voicer,YeahMaybe,$5,NaN,NaN,https://yeahmaybe.itch.io/the-choicer-voicer,https://itch.io/games/top-sellers,2026-10-13T12:00:00+00:00,5.00,https://yeahmaybe.itch.io/the-choicer-voicer,$5,NaN,https://itch.io/games/top-sellers
1,2,Martin: No Filter Fighters (Unofficial Fan Game).,Trey2k23,Free / no price displayed,Fighting,Play in browser,https://trey2k23.itch.io/martin-no-filter-figh...,https://itch.io/games/top-sellers,2026-10-13T12:00:00+00:00,0.00,https://trey2k23.itch.io/martin-no-filter-figh...,Free / no price displayed,Fighting,https://itch.io/games/top-sellers
2,3,The Annual Ghost Town Pumpkin Festival,adamgryu,Free / no price displayed,Simulation,NaN,https://adamgryu.itch.io/pumpkin-carver,https://itch.io/games/top-sellers,2026-10-13T12:00:00+00:00,0.00,https://adamgryu.itch.io/pumpkin-carver,Free / no price displayed,Simulation,https://itch.io/games/top-sellers
3,4,HYPERBALLIN' PLAYTEST,RED,Free / no price displayed,Platformer,NaN,https://redembergames.itch.io/hyperballin,https://itch.io/games/top-sellers,2026-10-13T12:00:00+00:00,0.00,https://redembergames.itch.io/hyperballin,Free / no price displayed,Platformer,https://itch.io/games/top-sellers
4,5,The Wrong Way,ZestosYt,$2.99,Adventure,NaN,https://zestosyt.itch.io/the-wrong-way,https://itch.io/games/top-sellers,2026-10-13T12:00:00+00:00,2.99,https://zestosyt.itch.io/the-wrong-way,$2.99,Adventure,https://itch.io/games/top-sellers


## 2. Clean fields and assess data quality

In [4]:
for c in ['title','url','developer','price_text','genres','platforms','source_page']:
    if c not in df: df[c]=pd.NA
    df[c]=df[c].astype('string').str.strip().replace({'':pd.NA,'None':pd.NA,'nan':pd.NA})
if 'rank' not in df: df['rank']=pd.NA
df['rank']=pd.to_numeric(df['rank'],errors='coerce')
if 'price_value' not in df:
    def parse_price(v):
        if pd.isna(v): return np.nan
        if 'free' in str(v).lower(): return 0.0
        m=re.search(r'(\d[\d,]*(?:\.\d{1,2})?)',str(v))
        return float(m.group(1).replace(',','')) if m else np.nan
    df['price_value']=df['price_text'].apply(parse_price)
df['price_value']=pd.to_numeric(df['price_value'],errors='coerce')
df['scraped_at']=pd.to_datetime(df.get('scraped_at'),errors='coerce',utc=True)
for c in ['genres','platforms']:
    df[c]=df[c].apply(lambda x:' | '.join(x) if isinstance(x,list) else x)
df=df.dropna(subset=['title','url']).drop_duplicates(['url','scraped_at'],keep='last').copy()
df['snapshot_date']=df['scraped_at'].dt.date.astype('string')
quality=pd.DataFrame({'dtype':df.dtypes.astype(str),'missing_count':df.isna().sum(),'missing_percent':(100*df.isna().mean()).round(1)})
display(quality); print('Unique games:',df.url.nunique(),'| snapshot dates:',sorted(df.snapshot_date.dropna().unique().tolist()))
display(df.groupby('snapshot_date',dropna=False).agg(observations=('url','size'),unique_games=('url','nunique')).reset_index())

,dtype,missing_count,missing_percent
rank,int64,0,0.0
title,string,0,0.0
developer,string,0,0.0
price,object,0,0.0
genre,object,337,9.6
platforms,object,2961,84.6
game_url,object,0,0.0
listing_page,object,0,0.0
scraped_at,"datetime64[ns, UTC]",0,0.0
price_value,float64,0,0.0


Unique games: 852 | snapshot dates: ['2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18', '2026-10-19']


,snapshot_date,observations,unique_games
0,2026-10-13,500,500
1,2026-10-14,500,500
2,2026-10-15,500,500
3,2026-10-16,500,500
4,2026-10-17,500,500
5,2026-10-18,500,500
6,2026-10-19,500,500
